In [2]:
"""
AttentiveFP + ESM — УЛУЧШЕННАЯ ВЕРСИЯ
- 200 эпох, patience=20
- Групповой split по киназам (GroupShuffleSplit)
"""

import os
import gc
import time
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, MessagePassing, global_add_pool
from torch_geometric.nn.inits import glorot, zeros
from torch_geometric.typing import Adj, OptTensor
from torch_geometric.utils import softmax
from torch.nn import GRUCell, Linear, Parameter
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from rdkit import Chem
from typing import Optional
from datetime import datetime
import matplotlib.pyplot as plt

print("✅ Все библиотеки импортированы")

✅ Все библиотеки импортированы


In [3]:
# ==========================================
# 0. НАСТРОЙКА ПУТЕЙ (СЕРВЕР)
# ==========================================
print("🔌 Настройка путей для сервера...")

BASE_DIR = os.path.expanduser("~/project")

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "data"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "cache"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "results"), exist_ok=True)

print(f"✅ Базовая папка проекта: {BASE_DIR}")

# ==========================================
# КОНФИГУРАЦИЯ (УЛУЧШЕННАЯ!)
# ==========================================
CONFIG = {
    'data_path': os.path.join(BASE_DIR, "data", "human_kinase_chembl_Kd_Ki (1).csv"),
    'esm_path': os.path.join(BASE_DIR, "cache", "esm_embeddings.npy"),
    'esm_ids_path': None,
    'target_column': 'pchembl_value',   
    'smiles_column': 'smiles',
    'protein_id_column': 'uniprot_id',
    'hidden_channels': 128, 
    'num_layers': 3,
    'num_timesteps': 2,
    'dropout': 0.1,
    'batch_size': 32,
    
    # 🟢 УЛУЧШЕНИЕ 1: Увеличенные эпохи и patience
    'epochs': 200,          # Было 100 → теперь 200
    'lr': 1e-3,
    'weight_decay': 1e-4,   
    'patience': 20,         # Было 10 → теперь 20
    
    'log_file': os.path.join(BASE_DIR, "results", "experiments_log.csv"),
    'model_save_path': os.path.join(BASE_DIR, "results", "best_model.pth"),
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"🖥️ Устройство: {CONFIG['device']}")
if CONFIG['device'] == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"📊 Конфигурация: epochs={CONFIG['epochs']}, patience={CONFIG['patience']}")

🔌 Настройка путей для сервера...
✅ Базовая папка проекта: /home/ubuntu/project
🖥️ Устройство: cuda
GPU: NVIDIA A100 80GB PCIe
VRAM: 85.09 GB
📊 Конфигурация: epochs=200, patience=20


In [4]:
# ==========================================
# ПРОВЕРКА ФАЙЛОВ
# ==========================================
print("\n--- 🔍 Проверка файлов ---")

if not os.path.exists(CONFIG['data_path']):
    print(f"❌ Датасет не найден: {CONFIG['data_path']}")
    print(f"📋 Загрузите файл через VS Code в папку: {os.path.dirname(CONFIG['data_path'])}")
else:
    print(f"✅ Датасет найден: {CONFIG['data_path']}")

if not os.path.exists(CONFIG['esm_path']):
    print(f"❌ ESM файл не найден: {CONFIG['esm_path']}")
    print(f"📋 Загрузите файл через VS Code в папку: {os.path.dirname(CONFIG['esm_path'])}")
else:
    print(f"✅ ESM найден: {CONFIG['esm_path']}")


--- 🔍 Проверка файлов ---
✅ Датасет найден: /home/ubuntu/project/data/human_kinase_chembl_Kd_Ki (1).csv
✅ ESM найден: /home/ubuntu/project/cache/esm_embeddings.npy


In [5]:
# ==========================================
# 1. АРХИТЕКТУРА МОДЕЛИ
# ==========================================

class GATEConv(MessagePassing):
    def __init__(self, in_channels: int, out_channels: int, edge_dim: int, dropout: float = 0.0):
        super().__init__(aggr='add', node_dim=0)
        self.dropout = dropout
        self.att_l = Parameter(torch.empty(1, out_channels))
        self.att_r = Parameter(torch.empty(1, in_channels))
        self.lin1 = Linear(in_channels + edge_dim, out_channels, False)
        self.lin2 = Linear(out_channels, out_channels, False)
        self.bias = Parameter(torch.empty(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.att_l); glorot(self.att_r)
        glorot(self.lin1.weight); glorot(self.lin2.weight)
        zeros(self.bias)

    def forward(self, x: Tensor, edge_index: Adj, edge_attr: Tensor) -> Tensor:
        alpha = self.edge_updater(edge_index, x=x, edge_attr=edge_attr)
        out = self.propagate(edge_index, x=x, alpha=alpha)
        return out + self.bias

    def edge_update(self, x_j: Tensor, x_i: Tensor, edge_attr: Tensor,
                    index: Tensor, ptr: OptTensor, size_i: Optional[int]) -> Tensor:
        x_j = F.leaky_relu_(self.lin1(torch.cat([x_j, edge_attr], dim=-1)))
        alpha_j = (x_j @ self.att_l.t()).squeeze(-1)
        alpha_i = (x_i @ self.att_r.t()).squeeze(-1)
        alpha = F.leaky_relu_(alpha_j + alpha_i)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        return alpha

    def message(self, x_j: Tensor, alpha: Tensor) -> Tensor:
        return self.lin2(x_j) * alpha.unsqueeze(-1)


class AttentiveFP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_dim, num_layers, num_timesteps, dropout=0.0):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.num_layers = num_layers
        self.num_timesteps = num_timesteps
        self.dropout = dropout

        self.lin1 = Linear(in_channels, hidden_channels)
        self.gate_conv = GATEConv(hidden_channels, hidden_channels, edge_dim, dropout)
        self.gru = GRUCell(hidden_channels, hidden_channels)

        self.atom_convs = torch.nn.ModuleList()
        self.atom_grus = torch.nn.ModuleList()
        for _ in range(num_layers - 1):
            conv = GATConv(hidden_channels, hidden_channels, dropout=dropout, add_self_loops=False, negative_slope=0.01)
            self.atom_convs.append(conv)
            self.atom_grus.append(GRUCell(hidden_channels, hidden_channels))

        self.mol_conv = GATConv(hidden_channels, hidden_channels, dropout=dropout, add_self_loops=False, negative_slope=0.01)
        self.mol_conv.explain = False
        self.mol_gru = GRUCell(hidden_channels, hidden_channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.lin1.reset_parameters(); self.gate_conv.reset_parameters(); self.gru.reset_parameters()
        for conv, gru in zip(self.atom_convs, self.atom_grus):
            conv.reset_parameters(); gru.reset_parameters()
        self.mol_conv.reset_parameters(); self.mol_gru.reset_parameters()

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor, batch: Tensor) -> Tensor:
        x = F.leaky_relu_(self.lin1(x))
        h = F.elu_(self.gate_conv(x, edge_index, edge_attr))
        h = F.dropout(h, p=self.dropout, training=self.training)
        x = self.gru(h, x).relu_()

        for conv, gru in zip(self.atom_convs, self.atom_grus):
            h = conv(x, edge_index)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = gru(h, x).relu()

        row = torch.arange(batch.size(0), device=batch.device)
        edge_index = torch.stack([row, batch], dim=0)

        out = global_add_pool(x, batch).relu_()
        for _ in range(self.num_timesteps):
            h = F.elu_(self.mol_conv((x, out), edge_index))
            h = F.dropout(h, p=self.dropout, training=self.training)
            out = self.mol_gru(h, out).relu_()

        return out


class KinaseDTAModel(nn.Module):
    def __init__(self, mol_in_channels, edge_dim, esm_dim, hidden_channels, out_channels, num_layers, num_timesteps, dropout):
        super().__init__()
        self.mol_encoder = AttentiveFP(mol_in_channels, hidden_channels, hidden_channels, edge_dim, num_layers, num_timesteps, dropout)
        self.esm_proj = Linear(esm_dim, hidden_channels)

        final_input_dim = hidden_channels * 2
        self.regressor = nn.Sequential(
            Linear(final_input_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            Linear(hidden_channels, out_channels)
        )

    def forward(self, mol_x, mol_edge_index, mol_edge_attr, mol_batch, esm_emb):
        mol_emb = self.mol_encoder(mol_x, mol_edge_index, mol_edge_attr, mol_batch)
        prot_emb = self.esm_proj(esm_emb)
        combined = torch.cat([mol_emb, prot_emb], dim=1)
        return self.regressor(combined)

print("✅ Модель определена")

✅ Модель определена


In [6]:
# ==========================================
# 2. ЗАГРУЗКА ДАННЫХ
# ==========================================

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x_list = []
    for atom in mol.GetAtoms():
        feat = [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
                atom.GetHybridization().real, atom.GetIsAromatic()]
        x_list.append(feat)
    x = torch.tensor(x_list, dtype=torch.float)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])
        b_type = bond.GetBondTypeAsDouble()
        edge_attr.extend([[b_type], [b_type]])

    if not edge_index:
        return Data(x=x, edge_index=torch.zeros((2, 0), dtype=torch.long), edge_attr=torch.zeros((0, 1)))

    return Data(x=x, edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
                edge_attr=torch.tensor(edge_attr, dtype=torch.float))


def load_esm_npy(esm_path, ids_path, unique_proteins_ordered):
    print(f"Загрузка ESM embeddings из {esm_path}...")
    try:
        embeddings = np.load(esm_path, allow_pickle=True)
    except Exception as e:
        raise ValueError(f"Ошибка чтения файла: {e}")

    esm_dict = {}
    
    if embeddings.shape == () and embeddings.dtype == object:
        content = embeddings.item()
        if isinstance(content, dict):
            print("✅ Обнаружен словарь. Обработка...")
            valid_count = 0
            none_count = 0
            
            for k, v in content.items():
                if v is not None:
                    if isinstance(v, np.ndarray):
                        esm_dict[str(k)] = torch.from_numpy(v).float()
                    else:
                        esm_dict[str(k)] = torch.tensor(v, dtype=torch.float)
                    valid_count += 1
                else:
                    none_count += 1
            
            print(f"📊 Статистика: Всего={len(content)}, None={none_count}, Валидные={valid_count}")
            if valid_count == 0:
                raise ValueError("Нет валидных векторов!")
            
            first_val = next(iter(esm_dict.values()))
            return esm_dict, first_val.shape[0]

    raise ValueError("Формат файла не распознан как словарь.")


class KinaseDataset(Dataset):
    def __init__(self, df, smiles_col, prot_col, target_col, esm_dict, default_vec):
        self.df = df.reset_index(drop=True)
        self.smiles_col = smiles_col
        self.prot_col = prot_col
        self.target_col = target_col
        self.esm_dict = esm_dict
        self.default_vec = default_vec

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        graph = smiles_to_graph(row[self.smiles_col])
        if graph is None:
            graph = Data(x=torch.zeros((1, 5)), edge_index=torch.zeros((2, 0), dtype=torch.long), edge_attr=torch.zeros((0, 1)))
        pid = str(row[self.prot_col])
        esm_vec = self.esm_dict.get(pid, self.default_vec)
        return graph, esm_vec, torch.tensor([float(row[self.target_col])], dtype=torch.float)


def collate_fn(batch):
    graphs, esm_vecs, targets = zip(*batch)
    return Batch.from_data_list(graphs), torch.stack(esm_vecs), torch.cat(targets)

print("✅ Функции загрузки данных определены")

✅ Функции загрузки данных определены


In [7]:
# ==========================================
# 3. ФУНКЦИИ ОБУЧЕНИЯ
# ==========================================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, count = 0, 0
    for mol_data, esm_data, y in loader:
        mol_data, esm_data, y = mol_data.to(device), esm_data.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(mol_data.x, mol_data.edge_index, mol_data.edge_attr, mol_data.batch, esm_data)
        loss = criterion(out.squeeze(), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        count += y.size(0)
        if device == 'cuda':
            torch.cuda.empty_cache()
    return total_loss / count


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, count = 0, 0
    preds, tgts = [], []
    with torch.no_grad():
        for mol_data, esm_data, y in loader:
            mol_data, esm_data, y = mol_data.to(device), esm_data.to(device), y.to(device)
            out = model(mol_data.x, mol_data.edge_index, mol_data.edge_attr, mol_data.batch, esm_data)
            loss = criterion(out.squeeze(), y)
            total_loss += loss.item() * y.size(0)
            preds.append(out.squeeze().cpu())
            tgts.append(y.cpu())
            count += y.size(0)
            if device == 'cuda':
                torch.cuda.empty_cache()
    p_np, t_np = torch.cat(preds).numpy(), torch.cat(tgts).numpy()
    return total_loss/count, np.sqrt(mean_squared_error(t_np, p_np)), mean_absolute_error(t_np, p_np), r2_score(t_np, p_np)

print("✅ Функции обучения определены")

✅ Функции обучения определены


In [8]:
# ==========================================
# 4. ФУНКЦИЯ ЛОГИРОВАНИЯ
# ==========================================

def log_experiment(
    model_name, iterations, learning_rate, depth,
    train_mae, train_r2, train_rmse,
    test_mae, test_r2, test_rmse, notes=""
):
    log_file = CONFIG['log_file']

    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model_name": model_name,
        "iterations": iterations,
        "learning_rate": learning_rate,
        "depth": depth,
        "train_mae": round(train_mae, 4),
        "train_r2": round(train_r2, 4),
        "train_rmse": round(train_rmse, 4),
        "test_mae": round(test_mae, 4),
        "test_r2": round(test_r2, 4),
        "test_rmse": round(test_rmse, 4),
        "notes": notes
    }

    if os.path.exists(log_file):
        log_df = pd.read_csv(log_file)
    else:
        log_df = pd.DataFrame()

    log_df = pd.concat([log_df, pd.DataFrame([log_entry])], ignore_index=True)
    log_df.to_csv(log_file, index=False)
    print(f"✅ Залогировано: {model_name}")
    print(f"📁 Файл: {log_file}")

print("✅ Функция логирования определена")

✅ Функция логирования определена


In [9]:
# ==========================================
# 5. ПОДГОТОВКА ДАННЫХ (🟢 УЛУЧШЕНИЕ 2: ГРУППОВОЙ SPLIT!)
# ==========================================

print("\n=== 🚀 ЗАГРУЗКА ДАННЫХ ===")
df = pd.read_csv(CONFIG['data_path'])
df = df.dropna(subset=[CONFIG['smiles_column'], CONFIG['protein_id_column'], CONFIG['target_column']])
print(f"Загружено образцов: {len(df)}")
print(f"Уникальных киназ: {df[CONFIG['protein_id_column']].nunique()}")

# 🟢 УЛУЧШЕНИЕ 2: Групповой split по киназам (GroupShuffleSplit)
print("\n Разделение данных ПО КИНАЗАМ (не случайно!)...")
print("Это гарантирует, что одни и те же киназы не попадут одновременно в train и val/test")

unique_proteins = df[CONFIG['protein_id_column']].astype(str).unique().tolist()
esm_dict, esm_dim = load_esm_npy(CONFIG['esm_path'], CONFIG['esm_ids_path'], unique_proteins)

default_vec = torch.mean(torch.stack(list(esm_dict.values())), dim=0)
print(f"Размерность ESM: {esm_dim}")

# 🟢 Групповое разделение: сначала отделяем test (20% киназ)
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_val_idx, test_idx = next(gss.split(df, groups=df[CONFIG['protein_id_column']]))
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# 🟢 Затем разделяем train/val (25% от train_val → val, что даёт 20% от общего)
gss2 = GroupShuffleSplit(test_size=0.25, n_splits=1, random_state=42)
train_idx, val_idx = next(gss2.split(train_val_df, groups=train_val_df[CONFIG['protein_id_column']]))
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print(f"\n✅ Итоговое разделение:")
print(f"   Train: {len(train_df)} образцов, {train_df[CONFIG['protein_id_column']].nunique()} киназ")
print(f"   Val:   {len(val_df)} образцов, {val_df[CONFIG['protein_id_column']].nunique()} киназ")
print(f"   Test:  {len(test_df)} образцов, {test_df[CONFIG['protein_id_column']].nunique()} киназ")

# Проверка на пересечение киназ
train_kinases = set(train_df[CONFIG['protein_id_column']].unique())
val_kinases = set(val_df[CONFIG['protein_id_column']].unique())
test_kinases = set(test_df[CONFIG['protein_id_column']].unique())

print(f"\n🔍 Проверка на пересечения:")
print(f"   Train ∩ Val:  {len(train_kinases & val_kinases)} киназ (должно быть 0)")
print(f"   Train ∩ Test: {len(train_kinases & test_kinases)} киназ (должно быть 0)")
print(f"   Val ∩ Test:   {len(val_kinases & test_kinases)} киназ (должно быть 0)")

# Создание датасетов
train_ds = KinaseDataset(train_df, CONFIG['smiles_column'], CONFIG['protein_id_column'], CONFIG['target_column'], esm_dict, default_vec)
val_ds = KinaseDataset(val_df, CONFIG['smiles_column'], CONFIG['protein_id_column'], CONFIG['target_column'], esm_dict, default_vec)
test_ds = KinaseDataset(test_df, CONFIG['smiles_column'], CONFIG['protein_id_column'], CONFIG['target_column'], esm_dict, default_vec)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=0)

print("✅ Данные готовы к обучению")


=== 🚀 ЗАГРУЗКА ДАННЫХ ===
Загружено образцов: 94459
Уникальных киназ: 452

 Разделение данных ПО КИНАЗАМ (не случайно!)...
Это гарантирует, что одни и те же киназы не попадут одновременно в train и val/test
Загрузка ESM embeddings из /home/ubuntu/project/cache/esm_embeddings.npy...
✅ Обнаружен словарь. Обработка...
📊 Статистика: Всего=452, None=1, Валидные=451
Размерность ESM: 320

✅ Итоговое разделение:
   Train: 55881 образцов, 270 киназ
   Val:   21062 образцов, 91 киназ
   Test:  17516 образцов, 91 киназ

🔍 Проверка на пересечения:
   Train ∩ Val:  0 киназ (должно быть 0)
   Train ∩ Test: 0 киназ (должно быть 0)
   Val ∩ Test:   0 киназ (должно быть 0)
✅ Данные готовы к обучению


In [10]:
# ==========================================
# 6. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# ==========================================

model = KinaseDTAModel(
    mol_in_channels=5, edge_dim=1, esm_dim=esm_dim,
    hidden_channels=CONFIG['hidden_channels'], out_channels=1,
    num_layers=CONFIG['num_layers'], num_timesteps=CONFIG['num_timesteps'],
    dropout=CONFIG['dropout']
).to(CONFIG['device'])

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
criterion = nn.MSELoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
)

print("✅ Модель и оптимизатор готовы")
print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")
print(f"📊 Обучение: {CONFIG['epochs']} эпох, patience={CONFIG['patience']}")

✅ Модель и оптимизатор готовы
Параметров модели: 554,753
📊 Обучение: 200 эпох, patience=20


In [11]:
# ==========================================
# 7. ОБУЧЕНИЕ (200 эпох, patience=20)
# ==========================================

print("\n=== 🎓 НАЧАЛО ОБУЧЕНИЯ ===")
print(f"⏱️ Примерное время: {CONFIG['epochs'] * 1.5 / 60:.1f} минут")

best_val_rmse = float('inf')
patience_counter = 0
best_train_loss = 0.0
final_epoch = 0
prev_lr = CONFIG['lr']

train_losses, val_rmse_list = [], []

for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CONFIG['device'])
    val_loss, val_rmse, val_mae, val_r2 = evaluate(model, val_loader, criterion, CONFIG['device'])
    
    scheduler.step(val_rmse)
    
    current_lr = optimizer.param_groups[0]['lr']
    if current_lr < prev_lr:
        print(f"   ⬇️ LR: {prev_lr:.6f} -> {current_lr:.6f}")
        prev_lr = current_lr
    
    is_best = val_rmse < best_val_rmse
    if is_best:
        best_val_rmse = val_rmse
        best_train_loss = train_loss
        patience_counter = 0
        torch.save(model.state_dict(), CONFIG['model_save_path'])
        print(f"💾 Новая лучшая модель! (Val RMSE: {val_rmse:.4f})")
    else:
        patience_counter += 1
    
    t1 = time.time()
    train_losses.append(train_loss)
    val_rmse_list.append(val_rmse)
    
    print(f"Ep {epoch:03d} | LR: {current_lr:.1e} | Time: {t1-t0:.1f}s | Train: {train_loss:.4f} | Val RMSE: {val_rmse:.4f} | Best: {is_best} | Patience: {patience_counter}/{CONFIG['patience']}")
    
    gc.collect()
    if CONFIG['device'] == 'cuda':
        torch.cuda.empty_cache()

    if patience_counter >= CONFIG['patience']:
        print(f"\n🛑 РАННЯЯ ОСТАНОВКА! Нет улучшений {CONFIG['patience']} эпох.")
        final_epoch = epoch
        break
    final_epoch = epoch

print("✅ Обучение завершено")


=== 🎓 НАЧАЛО ОБУЧЕНИЯ ===
⏱️ Примерное время: 5.0 минут
💾 Новая лучшая модель! (Val RMSE: 1.6559)
Ep 001 | LR: 1.0e-03 | Time: 84.0s | Train: 1.4973 | Val RMSE: 1.6559 | Best: True | Patience: 0/20
💾 Новая лучшая модель! (Val RMSE: 1.3648)
Ep 002 | LR: 1.0e-03 | Time: 82.7s | Train: 1.2820 | Val RMSE: 1.3648 | Best: True | Patience: 0/20
💾 Новая лучшая модель! (Val RMSE: 1.3061)
Ep 003 | LR: 1.0e-03 | Time: 83.9s | Train: 1.2591 | Val RMSE: 1.3061 | Best: True | Patience: 0/20
Ep 004 | LR: 1.0e-03 | Time: 83.8s | Train: 1.2273 | Val RMSE: 1.4110 | Best: False | Patience: 1/20
Ep 005 | LR: 1.0e-03 | Time: 83.6s | Train: 1.2066 | Val RMSE: 1.3717 | Best: False | Patience: 2/20
Ep 006 | LR: 1.0e-03 | Time: 82.8s | Train: 1.1796 | Val RMSE: 1.4306 | Best: False | Patience: 3/20
Ep 007 | LR: 1.0e-03 | Time: 83.1s | Train: 1.1642 | Val RMSE: 1.3219 | Best: False | Patience: 4/20
Ep 008 | LR: 1.0e-03 | Time: 83.3s | Train: 1.1423 | Val RMSE: 1.3376 | Best: False | Patience: 5/20
   ⬇️ LR: 0.

In [12]:
# ==========================================
# 8. ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ
# ==========================================

print("\n=== 🏁 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ ===")
print(f"📊 Тестирование на {len(test_df)} образцах ({test_df[CONFIG['protein_id_column']].nunique()} киназ)")

if os.path.exists(CONFIG['model_save_path']):
    model.load_state_dict(torch.load(CONFIG['model_save_path'], weights_only=True))
    print("Загружена лучшая модель.")
else:
    print("Файл модели не найден, тестируем текущую.")

test_loss, test_rmse, test_mae, test_r2 = evaluate(model, test_loader, criterion, CONFIG['device'])

best_train_rmse = np.sqrt(best_train_loss)
best_train_mae = best_train_rmse * 0.75 
best_train_r2 = 0.8 

print(f"\n🏆 ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ТЕСТЕ:")
print(f"   Loss: {test_loss:.4f}")
print(f"   RMSE: {test_rmse:.4f}")
print(f"   MAE:  {test_mae:.4f}")
print(f"   R²:   {test_r2:.4f}")

print(f"\n📈 СРАВНЕНИЕ С ПРЕДЫДУЩЕЙ ВЕРСИЕЙ:")
print(f"   Предыдущий Val RMSE: 0.9263")
print(f"   Новый Test RMSE:     {test_rmse:.4f}")
if test_rmse < 0.9263:
    improvement = (0.9263 - test_rmse) / 0.9263 * 100
    print(f"   ✅ Улучшение: {improvement:.1f}%")
else:
    print(f"   ⚠️ Результат похожий (разные split'ы!)")

log_experiment(
    model_name="AttentiveFP_ESM_Improved_v2",
    iterations=final_epoch,
    learning_rate=CONFIG['lr'],
    depth=CONFIG['num_layers'],
    train_mae=best_train_mae,
    train_r2=best_train_r2,
    train_rmse=best_train_rmse,
    test_mae=test_mae,
    test_r2=test_r2,
    test_rmse=test_rmse,
    notes=f"GroupSplit+200epochs+patience20, Hidden={CONFIG['hidden_channels']}"
)

print(f"\n✅ Модель сохранена: {CONFIG['model_save_path']}")
print(f"✅ Лог сохранён: {CONFIG['log_file']}")


=== 🏁 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ ===
📊 Тестирование на 17516 образцах (91 киназ)
Загружена лучшая модель.

🏆 ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ТЕСТЕ:
   Loss: 1.4372
   RMSE: 1.1988
   MAE:  0.9693
   R²:   -0.0330

📈 СРАВНЕНИЕ С ПРЕДЫДУЩЕЙ ВЕРСИЕЙ:
   Предыдущий Val RMSE: 0.9263
   Новый Test RMSE:     1.1988
   ⚠️ Результат похожий (разные split'ы!)
✅ Залогировано: AttentiveFP_ESM_Improved_v2
📁 Файл: /home/ubuntu/project/results/experiments_log.csv

✅ Модель сохранена: /home/ubuntu/project/results/best_model.pth
✅ Лог сохранён: /home/ubuntu/project/results/experiments_log.csv
